<a href="https://colab.research.google.com/github/laharipallekala/bangalore-aqi-project/blob/main/Bengaluru_AQI_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Phase 1. Data Collection

In [1]:
# Install all libraries needed for Phase 1
# The ! prefix tells Colab to run this as a terminal command, not Python
!pip install pandas numpy requests matplotlib seaborn openpyxl --quiet
# Confirm they loaded correctly
import pandas as pd
import numpy as np
import requests
print('All libraries loaded successfully')
print('Pandas version:', pd.__version__)

All libraries loaded successfully
Pandas version: 2.2.2


In [2]:
#KGAT_50f7051de1c61da491ce9e4468d115eb
import os, json   # os handles file system, json handles JSON files

# Create the hidden .kaggle directory that the Kaggle library expects
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)   # ~ means home directory

# Build the credentials dictionary
token = {
    "username": "lahariii2006",
    "key": "KGAT_50f7051de1c61da491ce9e4468d115eb"
}

# Write it as a JSON file
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(token, f)   # saves the dictionary as a .json file

# Set file permissions so only you can read it (security requirement)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)   # 0o600 = ownerread/write only
print("Kaggle credentials saved successfully")

Kaggle credentials saved successfully


In [3]:
import zipfile, glob   # zipfile extracts zip archives, glob finds files by pattern

# Download — Kaggle saves the zip to the current directory (/content in Colab)
!kaggle datasets download -d rohanrao/air-quality-data-in-india

# Find and extract all zip files that were downloaded
for f in glob.glob('*.zip'):   # glob('*.zip') finds all .zip files in current folder
    with zipfile.ZipFile(f, 'r') as z:   # open the zip in read mode
        z.extractall('./raw_data')   # extract everything into raw_data folder
        print(f'Extracted: {f}')

# Show all extracted files
print('Files in raw_data:')
print(os.listdir('./raw_data'))

Dataset URL: https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india
License(s): CC0-1.0
100% 72.9M/72.9M [00:00<00:00, 116MB/s]

Extracted: air-quality-data-in-india.zip
Files in raw_data:
['city_day.csv', 'station_hour.csv', 'city_hour.csv', 'station_day.csv', 'stations.csv']


In [4]:
import pandas as pd

# Read the CSV file into a DataFrame
df_raw = pd.read_csv('./raw_data/city_day.csv')   # pd.read_csv reads a CSV file into a table

# Shape = (rows, columns) — tells you how large the dataset is
print('Shape:', df_raw.shape)   # expected: ~29531 rows, 16 columns

# Columns = the header names of each column
print('Columns:', df_raw.columns.tolist())

# Head = first 5 rows — gives you a feel for the data
print(df_raw.head())

# Show all unique city names so you know what to filter for
print('Cities:', sorted(df_raw['City'].unique()))   # look for Bengaluru in this list

Shape: (29531, 16)
Columns: ['City', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket']
        City        Date  PM2.5  PM10     NO    NO2    NOx  NH3     CO    SO2  \
0  Ahmedabad  2015-01-01    NaN   NaN   0.92  18.22  17.15  NaN   0.92  27.64   
1  Ahmedabad  2015-01-02    NaN   NaN   0.97  15.69  16.46  NaN   0.97  24.55   
2  Ahmedabad  2015-01-03    NaN   NaN  17.40  19.30  29.70  NaN  17.40  29.07   
3  Ahmedabad  2015-01-04    NaN   NaN   1.70  18.48  17.97  NaN   1.70  18.59   
4  Ahmedabad  2015-01-05    NaN   NaN  22.10  21.42  37.76  NaN  22.10  39.33   

       O3  Benzene  Toluene  Xylene  AQI AQI_Bucket  
0  133.36     0.00     0.02    0.00  NaN        NaN  
1   34.06     3.68     5.50    3.77  NaN        NaN  
2   30.70     6.80    16.40    2.25  NaN        NaN  
3   36.08     4.43    10.14    1.00  NaN        NaN  
4   39.31     7.01    18.89    2.78  NaN        NaN  
Cities: ['Ahmedabad', 'Aizaw

In [5]:
# str.lower() converts to lowercase first so we catch all spelling variants
# str.contains() checks if the string contains our search term
# na=False means: if the cell is empty, treat it as 'not matching'
bangalore1 = df_raw[df_raw['City'].str.lower()
                   .str.contains('bangalore|bengaluru', na=False)]   # | means OR

print('Bangalore rows:', len(bangalore1))   # expected: ~1800 rows
print('Date range:', bangalore1['Date'].min(), '->', bangalore1['Date'].max())
print('Missing AQI %:', round(bangalore1['AQI'].isnull().mean() * 100, 1), '%')
print('Missing PM2.5 %:', round(bangalore1['PM2.5'].isnull().mean() * 100, 1),
'%')

# Keep only the columns we need for the master CSV
bangalore1 = bangalore1[['Date', 'AQI', 'PM2.5', 'PM10']].copy()   # .copy() prevents a SettingWithCopyWarning
bangalore1['Date'] = pd.to_datetime(bangalore1['Date'])   # convert Date to proper datetime type
bangalore1 = bangalore1.sort_values('Date').reset_index(drop=True)   # sort chronologically, reset row numbers
print('Dataset 1 ready:', bangalore1.shape)

Bangalore rows: 2009
Date range: 2015-01-01 -> 2020-07-01
Missing AQI %: 4.9 %
Missing PM2.5 %: 7.3 %
Dataset 1 ready: (2009, 4)


In [6]:
import pandas as pd

df_kaggle1 = pd.read_csv('./raw_data/city_day.csv')
bangalore1 = df_kaggle1[df_kaggle1['City'].str.lower().str.contains('bangalore|bengaluru', na=False)].copy()
bangalore1 = bangalore1[['Date', 'AQI', 'PM2.5', 'PM10']].copy()
bangalore1['Date'] = pd.to_datetime(bangalore1['Date'])
bangalore1 = bangalore1.sort_values('Date').reset_index(drop=True)

print('Dataset 1 ready:', bangalore1.shape)
print('Date range:', bangalore1['Date'].min().date(), '->', bangalore1['Date'].max().date())
print('Missing %:')
print((bangalore1.isnull().sum() / len(bangalore1) * 100).round(1))

Dataset 1 ready: (2009, 4)
Date range: 2015-01-01 -> 2020-07-01
Missing %:
Date      0.0
AQI       4.9
PM2.5     7.3
PM10     17.9
dtype: float64


In [7]:
"""
#API: b6633b140e5781661e3a90dfec38fde708e6cd89
import requests

# Paste your token here once you get it
TOKEN = 'b6633b140e5781661e3a90dfec38fde708e6cd89'

# Test with Bengaluru city feed
resp = requests.get(f'https://api.waqi.info/feed/bengaluru/?token={TOKEN}')
print('Status:', resp.status_code)
data = resp.json()
print('API status:', data['status'])

if data['status'] == 'ok':
    d = data['data']
    print('City:', d.get('city', {}).get('name'))
    print('AQI:', d['aqi'])
    print('PM2.5:', d['iaqi'].get('pm25', {}).get('v'))
    print('PM10:', d['iaqi'].get('pm10', {}).get('v'))
    print('Time:', d['time']['s'])
    """

"\n#API: b6633b140e5781661e3a90dfec38fde708e6cd89\nimport requests\n\n# Paste your token here once you get it\nTOKEN = 'b6633b140e5781661e3a90dfec38fde708e6cd89'\n\n# Test with Bengaluru city feed\nresp = requests.get(f'https://api.waqi.info/feed/bengaluru/?token={TOKEN}')\nprint('Status:', resp.status_code)\ndata = resp.json()\nprint('API status:', data['status'])\n\nif data['status'] == 'ok':\n    d = data['data']\n    print('City:', d.get('city', {}).get('name'))\n    print('AQI:', d['aqi'])\n    print('PM2.5:', d['iaqi'].get('pm25', {}).get('v'))\n    print('PM10:', d['iaqi'].get('pm10', {}).get('v'))\n    print('Time:', d['time']['s'])\n    "

In [8]:
"""
# Search for all Bengaluru stations
resp = requests.get(f'https://api.waqi.info/search/?token={TOKEN}&keyword=bengaluru')
stations = resp.json().get('data', [])

print(f'Found {len(stations)} stations\n')
for s in stations:
    print(f"UID: {s['uid']} | Name: {s['station']['name']} | AQI: {s['aqi']}")
    """


'\n# Search for all Bengaluru stations\nresp = requests.get(f\'https://api.waqi.info/search/?token={TOKEN}&keyword=bengaluru\')\nstations = resp.json().get(\'data\', [])\n\nprint(f\'Found {len(stations)} stations\n\')\nfor s in stations:\n    print(f"UID: {s[\'uid\']} | Name: {s[\'station\'][\'name\']} | AQI: {s[\'aqi\']}")\n    '

In [9]:
"""
import requests, pandas as pd, time

TOKEN = 'b6633b140e5781661e3a90dfec38fde708e6cd89'

station_uids = [11293, 11276, 11270, 11312, 11428, 8686, 8190, 12441, 8687, 8191]

all_records = []

for uid in station_uids:
    url = f'https://api.waqi.info/feed/@{uid}/?token={TOKEN}'
    resp = requests.get(url)
    data = resp.json()

    if data['status'] != 'ok':
        print(f'UID {uid}: skipping — status {data["status"]}')
        continue

    d = data['data']
    station_name = d.get('city', {}).get('name', f'Station {uid}')

    # Get current reading
    record = {
        'station_uid': uid,
        'station_name': station_name,
        'Date': d['time']['s'][:10],
        'AQI': d.get('aqi'),
        'PM2.5': d['iaqi'].get('pm25', {}).get('v'),
        'PM10': d['iaqi'].get('pm10', {}).get('v'),
        'humidity': d['iaqi'].get('h', {}).get('v'),
        'temp': d['iaqi'].get('t', {}).get('v'),
        'wind_speed': d['iaqi'].get('w', {}).get('v'),
    }
    all_records.append(record)
    print(f"UID {uid} | {station_name} | AQI={record['AQI']} | PM2.5={record['PM2.5']}")
    time.sleep(0.5)

df_aqicn = pd.DataFrame(all_records)
print('\nShape:', df_aqicn.shape)
print(df_aqicn)
"""

'\nimport requests, pandas as pd, time\n\nTOKEN = \'b6633b140e5781661e3a90dfec38fde708e6cd89\'\n\nstation_uids = [11293, 11276, 11270, 11312, 11428, 8686, 8190, 12441, 8687, 8191]\n\nall_records = []\n\nfor uid in station_uids:\n    url = f\'https://api.waqi.info/feed/@{uid}/?token={TOKEN}\'\n    resp = requests.get(url)\n    data = resp.json()\n\n    if data[\'status\'] != \'ok\':\n        print(f\'UID {uid}: skipping — status {data["status"]}\')\n        continue\n\n    d = data[\'data\']\n    station_name = d.get(\'city\', {}).get(\'name\', f\'Station {uid}\')\n\n    # Get current reading\n    record = {\n        \'station_uid\': uid,\n        \'station_name\': station_name,\n        \'Date\': d[\'time\'][\'s\'][:10],\n        \'AQI\': d.get(\'aqi\'),\n        \'PM2.5\': d[\'iaqi\'].get(\'pm25\', {}).get(\'v\'),\n        \'PM10\': d[\'iaqi\'].get(\'pm10\', {}).get(\'v\'),\n        \'humidity\': d[\'iaqi\'].get(\'h\', {}).get(\'v\'),\n        \'temp\': d[\'iaqi\'].get(\'t\', {}).get(

In [10]:
"""
import requests, pandas as pd, time
from datetime import datetime, timedelta

TOKEN = 'b6633b140e5781661e3a90dfec38fde708e6cd89'

# Using only stations that have PM2.5 data — skip 8686, 8687, 8191
valid_stations = {
    11293: 'Silk Board',
    11276: 'Jayanagar 5th Block',
    11270: 'Hombegowda Nagar',
    11312: 'Bapuji Nagar',
    11428: 'Hebbal',
    8190:  'BTM',
    12441: 'BWSSB Kadabesanahalli',
}

all_records = []

for uid, name in valid_stations.items():
    print(f'Fetching history for {name} (UID {uid})...')

    url = f'https://api.waqi.info/feed/@{uid}/?token={TOKEN}'
    resp = requests.get(url)

    if resp.json()['status'] != 'ok':
        print(f'  Skipping — bad status')
        continue

    d = resp.json()['data']

    # Extract historical forecast/history data embedded in the feed
    # aqicn embeds up to 7 days of history in the 'forecast' field
    forecast = d.get('forecast', {}).get('daily', {})

    # Also get the historical data from the debug endpoint
    hist_url = f'https://api.waqi.info/api/feed/@{uid}/hist.json?token={TOKEN}'
    hist_resp = requests.get(hist_url)

    print(f'  Current: AQI={d.get("aqi")} PM2.5={d["iaqi"].get("pm25",{}).get("v")}')
    print(f'  Forecast keys available: {list(forecast.keys())}')

    # Add current reading
    all_records.append({
        'station_uid': uid,
        'station_name': name,
        'Date': d['time']['s'][:10],
        'AQI': d.get('aqi'),
        'PM2.5': d['iaqi'].get('pm25', {}).get('v'),
        'PM10': d['iaqi'].get('pm10', {}).get('v'),
        'humidity': d['iaqi'].get('h', {}).get('v'),
        'temp': d['iaqi'].get('t', {}).get('v'),
        'wind_speed': d['iaqi'].get('w', {}).get('v'),
    })

    # Extract any historical pm25 data from forecast daily
    if 'pm25' in forecast:
        for entry in forecast['pm25']:
            all_records.append({
                'station_uid': uid,
                'station_name': name,
                'Date': entry['day'],
                'PM2.5': entry.get('avg'),
                'AQI': None,
                'PM10': None,
                'humidity': None,
                'temp': None,
                'wind_speed': None,
            })

    time.sleep(1)

df_hist = pd.DataFrame(all_records)
df_hist['Date'] = pd.to_datetime(df_hist['Date'])
df_hist = df_hist.sort_values(['station_uid','Date'])

print('\nAll records shape:', df_hist.shape)
print('Date range:', df_hist['Date'].min().date(), '->', df_hist['Date'].max().date())
print(df_hist[['station_name','Date','AQI','PM2.5']].head(20))
"""

'\nimport requests, pandas as pd, time\nfrom datetime import datetime, timedelta\n\nTOKEN = \'b6633b140e5781661e3a90dfec38fde708e6cd89\'\n\n# Using only stations that have PM2.5 data — skip 8686, 8687, 8191\nvalid_stations = {\n    11293: \'Silk Board\',\n    11276: \'Jayanagar 5th Block\',\n    11270: \'Hombegowda Nagar\',\n    11312: \'Bapuji Nagar\',\n    11428: \'Hebbal\',\n    8190:  \'BTM\',\n    12441: \'BWSSB Kadabesanahalli\',\n}\n\nall_records = []\n\nfor uid, name in valid_stations.items():\n    print(f\'Fetching history for {name} (UID {uid})...\')\n\n    url = f\'https://api.waqi.info/feed/@{uid}/?token={TOKEN}\'\n    resp = requests.get(url)\n\n    if resp.json()[\'status\'] != \'ok\':\n        print(f\'  Skipping — bad status\')\n        continue\n\n    d = resp.json()[\'data\']\n\n    # Extract historical forecast/history data embedded in the feed\n    # aqicn embeds up to 7 days of history in the \'forecast\' field\n    forecast = d.get(\'forecast\', {}).get(\'daily\',

In [11]:
import requests, pandas as pd

# Date range comes directly from your bangalore1 data
start_date = bangalore1['Date'].min().strftime('%Y-%m-%d')
end_date   = bangalore1['Date'].max().strftime('%Y-%m-%d')
print(f'Fetching weather: {start_date} to {end_date}')

resp = requests.get('https://archive-api.open-meteo.com/v1/archive', params={
    'latitude':    12.97,
    'longitude':   77.59,
    'start_date':  start_date,
    'end_date':    end_date,
    'daily': [
        'temperature_2m_mean',
        'precipitation_sum',
        'windspeed_10m_max',
        'relative_humidity_2m_mean'
    ],
    'timezone': 'Asia/Kolkata'
})

print('API status:', resp.status_code)

weather = pd.DataFrame(resp.json()['daily'])
weather.rename(columns={
    'time':                       'Date',
    'temperature_2m_mean':        'temp',
    'precipitation_sum':          'rainfall',
    'windspeed_10m_max':          'wind_speed',
    'relative_humidity_2m_mean':  'humidity'
}, inplace=True)

weather['Date'] = pd.to_datetime(weather['Date'])
weather.to_csv('./raw_data/weather_bengaluru.csv', index=False)
print('Weather saved. Shape:', weather.shape)
print(weather.head())

Fetching weather: 2015-01-01 to 2020-07-01
API status: 200
Weather saved. Shape: (2009, 5)
        Date  temp  rainfall  wind_speed  humidity
0 2015-01-01  21.7       4.8        13.2        79
1 2015-01-02  21.7       0.8        13.0        75
2 2015-01-03  21.5       0.0        10.5        71
3 2015-01-04  21.3       0.0        11.4        70
4 2015-01-05  21.7       0.0        11.9        69


In [12]:
# Merge AQI + weather on Date
master = pd.merge(bangalore1, weather, on='Date', how='left')
print('After merge shape:', master.shape)
print('Missing after merge:')
print(master.isnull().sum())

# Fill any small weather gaps
for col in ['temp', 'humidity', 'wind_speed', 'rainfall']:
    master[col] = master[col].fillna(method='ffill', limit=3)
    master[col] = master[col].fillna(method='bfill', limit=3)

# Fill AQI/PM2.5/PM10 gaps
for col in ['AQI', 'PM2.5', 'PM10']:
    master[col] = master[col].fillna(method='ffill', limit=7)
    master[col] = master[col].fillna(method='bfill', limit=7)
    master['_m'] = master['Date'].dt.month
    master[col]  = master[col].fillna(master.groupby('_m')[col].transform('mean'))
master.drop(columns=['_m'], inplace=True)

# Add engineered columns
def get_season(month):
    if month in [12, 1, 2]:      return 'Winter'
    elif month in [3, 4, 5]:     return 'Summer'
    elif month in [6, 7, 8, 9]:  return 'Monsoon'
    else:                         return 'Post-Monsoon'

master['year']        = master['Date'].dt.year
master['month']       = master['Date'].dt.month
master['day_of_week'] = master['Date'].dt.day_name()
master['season']      = master['month'].apply(get_season)

# Final column order
final_cols = ['Date', 'year', 'month', 'season', 'day_of_week',
              'AQI', 'PM2.5', 'PM10', 'temp', 'humidity', 'wind_speed', 'rainfall']
master = master[final_cols]

print('\nMaster shape:', master.shape)
print('Missing values:')
print(master.isnull().sum())
print('\nSample:')
print(master.head())

After merge shape: (2009, 8)
Missing after merge:
Date            0
AQI            99
PM2.5         146
PM10          360
temp            0
rainfall        0
wind_speed      0
humidity        0
dtype: int64

Master shape: (2009, 12)
Missing values:
Date           0
year           0
month          0
season         0
day_of_week    0
AQI            0
PM2.5          0
PM10           0
temp           0
humidity       0
wind_speed     0
rainfall       0
dtype: int64

Sample:
        Date  year  month  season day_of_week         AQI      PM2.5  \
0 2015-01-01  2015      1  Winter    Thursday  111.729032  46.894452   
1 2015-01-02  2015      1  Winter      Friday  111.729032  46.894452   
2 2015-01-03  2015      1  Winter    Saturday  111.729032  46.894452   
3 2015-01-04  2015      1  Winter      Sunday  111.729032  46.894452   
4 2015-01-05  2015      1  Winter      Monday  111.729032  46.894452   

         PM10  temp  humidity  wind_speed  rainfall  
0  101.797484  21.7        79        1

/tmp/ipykernel_10617/148205032.py:9: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  master[col] = master[col].fillna(method='ffill', limit=3)
/tmp/ipykernel_10617/148205032.py:10: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  master[col] = master[col].fillna(method='bfill', limit=3)
/tmp/ipykernel_10617/148205032.py:14: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  master[col] = master[col].fillna(method='ffill', limit=7)
/tmp/ipykernel_10617/148205032.py:15: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  master[col] = master[col].fillna(method='bfill', limit=7)
/tmp/ipykernel_10617/148205032.py:14: FutureWarning: Series.fillna with 'meth

In [17]:
import os

# Validation
print('='*50)
print('VALIDATION REPORT')
print('='*50)
print(f'Rows: {master.shape[0]} (minimum required: 730)')
print(f'Columns: {master.shape[1]}')
print(f'Date range: {master["Date"].min().date()} -> {master["Date"].max().date()}')
print(f'Duplicate dates: {master.duplicated(subset="Date").sum()}')
print(f'Season distribution:\n{master["season"].value_counts()}')
print(f'\nMissing values:\n{master.isnull().sum()}')
print('\nPASS' if master.shape[0] >= 730 and master.isnull().sum().sum() == 0 else '\nREVIEW NEEDED')

# Create the 'processed' directory if it doesn't exist
os.makedirs('./processed', exist_ok=True)

# Save
master.to_csv('./processed/bangalore_master_aqi.csv', index=False)
print('\nSaved: bangalore_master_aqi.csv')

# Download to your computer
from google.colab import files
files.download('./processed/bangalore_master_aqi.csv')

VALIDATION REPORT
Rows: 2009 (minimum required: 730)
Columns: 12
Date range: 2015-01-01 -> 2020-07-01
Duplicate dates: 0
Season distribution:
season
Monsoon         641
Summer          552
Winter          511
Post-Monsoon    305
Name: count, dtype: int64

Missing values:
Date           0
year           0
month          0
season         0
day_of_week    0
AQI            0
PM2.5          0
PM10           0
temp           0
humidity       0
wind_speed     0
rainfall       0
dtype: int64

PASS

Saved: bangalore_master_aqi.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
print('Shape before outlier removal:', master.shape)

# AQI valid range: 0-500
before = len(master)
master = master[master['AQI'].between(0, 500)]
print(f'AQI outliers removed: {before - len(master)}')

# PM2.5 valid range: 0-500
before = len(master)
master = master[master['PM2.5'].between(0, 500)]
print(f'PM2.5 outliers removed: {before - len(master)}')

# PM10 valid range: 0-600
before = len(master)
master = master[master['PM10'].between(0, 600)]
print(f'PM10 outliers removed: {before - len(master)}')

# Temperature valid range for Bengaluru: 10-45°C
before = len(master)
master = master[master['temp'].between(10, 45)]
print(f'Temp outliers removed: {before - len(master)}')

# Humidity: 0-100%
before = len(master)
master = master[master['humidity'].between(0, 100)]
print(f'Humidity outliers removed: {before - len(master)}')

# Wind speed: 0-150 km/h
before = len(master)
master = master[master['wind_speed'].between(0, 150)]
print(f'Wind speed outliers removed: {before - len(master)}')

# Rainfall: 0-500mm
before = len(master)
master = master[master['rainfall'].between(0, 500)]
print(f'Rainfall outliers removed: {before - len(master)}')

print('\nShape after outlier removal:', master.shape)

Shape before outlier removal: (2009, 12)
AQI outliers removed: 0
PM2.5 outliers removed: 0
PM10 outliers removed: 0
Temp outliers removed: 0
Humidity outliers removed: 0
Wind speed outliers removed: 0
Rainfall outliers removed: 0

Shape after outlier removal: (2009, 12)


In [18]:
master.to_csv('./processed/bangalore_master_aqi.csv', index=False)
print('Master CSV re-saved.')
print('Final shape:', master.shape)
print('Missing values:', master.isnull().sum().sum())

from google.colab import files
files.download('./processed/bangalore_master_aqi.csv')

Master CSV re-saved.
Final shape: (2009, 12)
Missing values: 0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Phase 2: Exploratory Data Analysis (EDA)